#### Testing generated market level data such that there is no endogeonous spending and spending within geo is uncorrelated across channels

In [ ]:
import numpy as np
import pandas as pd 

from meridian import constants 
from meridian.data import load, test_utils 
from meridian.model import model, spec, prior_distribution 
from meridian.analysis import optimizer, analyzer, visualizer, summarizer, formatter 

import tensorflow as tf 
import tensorflow_probability as tfp

%matplotlib inline
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print(f"Your runtime has {ram_gb:.1f} gigabytes of available RAM\n")

import sys
print(sys.executable)


In [ ]:
from mmm_lab.data_generation.baseline import generate_baseline_geo_data
from mmm_lab.data_generation.marketing import add_marketing_effects, geometric_adstock, hill_saturation


# Set seed for reproducibility
np.random.seed(42)

# Generate baseline data
baseline_df = generate_baseline_geo_data(n_geos=40, n_weeks=104, start_date='2023-01-01')
df = add_marketing_effects(baseline_df, channels=['tv', 'paid_search'])

df['total_bookings'] = df['baseline_bookings'] + df['effect_tv'] + df['effect_paid_search']

# Verify media share
media_mean = (df['effect_tv'] + df['effect_paid_search']).mean()
total_mean = df['total_bookings'].mean()
print(f"Media share: {media_mean/total_mean:.1%}")

print(f"\nData shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Columns: {df.columns.tolist()}")

# Store ground truth for validation
ground_truth = df.attrs['channel_params']
print("\n" + "="*80)
print("GROUND TRUTH PARAMETERS (What we're trying to recover)")
print("="*80)
for channel, params in ground_truth.items():
    print(f"\n{channel.upper()}:")
    for k, v in params.items():
        print(f"  {k}: {v}")

In [ ]:
# Key diagnostic: cross-sectional variation in per-capita spend and effects
geo_summary = df.groupby('geo').agg(
    population=('population', 'first'),
    spend_tv=('spend_tv', 'sum'),
    spend_ps=('spend_paid_search', 'sum'),
    effect_tv=('effect_tv', 'sum'),
    effect_ps=('effect_paid_search', 'sum'),
    total_bookings=('total_bookings', 'sum'),
).reset_index()

for col in ['spend_tv', 'spend_ps', 'effect_tv', 'effect_ps', 'total_bookings']:
    geo_summary[f'{col}_pc'] = geo_summary[col] / geo_summary['population']

print("=== Cross-Sectional Variation (Per Capita) ===")
for col, label in [('spend_tv_pc', 'TV spend'), ('effect_tv_pc', 'TV effect'),
                    ('spend_ps_pc', 'PS spend'), ('effect_ps_pc', 'PS effect')]:
    cv = geo_summary[col].std() / geo_summary[col].mean()
    print(f"{label:20s} CV: {cv:.2%}")

print("\n=== Spend → Effect Correlations (Cross-Sectional) ===")
for ch in ['tv', 'ps']:
    corr = np.corrcoef(geo_summary[f'spend_{ch}_pc'], geo_summary[f'effect_{ch}_pc'])[0,1]
    print(f"{ch.upper():5s} per-capita spend vs effect: r = {corr:.3f}")


In [ ]:
national_truth_tv = df['effect_tv'].sum() / df['spend_tv'].sum()
national_truth_ps = df['effect_paid_search'].sum() / df['spend_paid_search'].sum()

print(f"National ground truth ROAS:")
print(f"  TV:          {national_truth_tv:.3f}")
print(f"  Paid Search: {national_truth_ps:.3f}")

from numpy.linalg import lstsq

n = len(df)

# Lag spend by 1 week (within geo - need to handle geo boundaries)
df_sorted = df.sort_values(['geo', 'date'])
tv_lag1 = df_sorted.groupby('geo')['spend_tv'].shift(1).fillna(0).values
ps_lag1 = df_sorted.groupby('geo')['spend_paid_search'].shift(1).fillna(0).values

X = np.column_stack([
    np.ones(n),
    df_sorted['demand_good'].values,
    df_sorted['spend_tv'].values,
    tv_lag1,
    df_sorted['spend_paid_search'].values,
    ps_lag1,
])
y_ols = df_sorted['total_bookings'].values

coefs, _, _, _ = lstsq(X, y_ols, rcond=None)

# ROAS = sum of current + lagged coefficients
ols_roas_tv = coefs[2] + coefs[3]
ols_roas_ps = coefs[4] + coefs[5]

print("=== OLS with 1-period lag ===")
print(f"TV:  current={coefs[2]:.4f}, lag1={coefs[3]:.4f}, total ROAS={ols_roas_tv:.3f} (true: {national_truth_tv:.3f})")
print(f"PS:  current={coefs[4]:.4f}, lag1={coefs[5]:.4f}, total ROAS={ols_roas_ps:.3f} (true: {national_truth_ps:.3f})")


In [ ]:
# Meridian expects specific column structure
# We'll keep it simple for MVP: no media impressions, just spend

meridian_df = df[[
    'date', 'geo', 'population',
    'total_bookings',  # This will be our KPI
    'spend_tv', 'spend_paid_search',
    'demand_good', 'demand_perfect', 'demand_poor'
]].copy()

print(f"\nMeridian data shape: {meridian_df.shape}")
print(f"\nColumns: {meridian_df.columns.tolist()}")
print(f"\nFirst few rows:")
print(meridian_df.head())

# Save to CSV (Meridian requirement)
csv_path = 'data/simulated_mmm_data.csv'
meridian_df.to_csv(csv_path, index=False)
print(f"\n✓ Data saved to: {csv_path}")

In [ ]:
# Map DataFrame columns to Meridian concepts
coord_to_columns = load.CoordToColumns(
    time='date',
    geo='geo',
    population='population',
    kpi='total_bookings',
    
    # For MVP, we'll use spend as proxy for impressions
    # (Meridian prefers impressions but can work with spend)
    media=['spend_tv', 'spend_paid_search'],
    media_spend=['spend_tv', 'spend_paid_search'],
    
    # Optional: controls for demand
    controls=['demand_good'],  
#    controls=[],  # Test no controls first 
)

# Channel naming (Meridian likes clean names)
media_to_channel = {
    'spend_tv': 'TV',
    'spend_paid_search': 'Paid_Search'
}

media_spend_to_channel = {
    'spend_tv': 'TV',
    'spend_paid_search': 'Paid_Search'
}

print("✓ Coordinate mappings defined")

In [ ]:
# Load data using Meridian's CSV loader
loader = load.CsvDataLoader(
    csv_path=csv_path,
    kpi_type='revenue',  # Treat 'bookings' as revenue for simplicity 
    coord_to_columns=coord_to_columns,
    media_to_channel=media_to_channel,
    media_spend_to_channel=media_spend_to_channel
)

data = loader.load()

In [ ]:
# Set median = breakeven ROAS for each channel, with a moderate sigma to allow for uncertainty
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(
        loc=tf.constant([np.log(1.0), np.log(1.0)], dtype=tf.float32),
        scale=tf.constant([0.3, 0.3], dtype=tf.float32),
        name=constants.ROI_M,
    ),
)


model_spec = spec.ModelSpec(
    prior=prior
)

In [ ]:
# Initialize model
mmm = model.Meridian(
    input_data=data,
    model_spec=model_spec
)



In [ ]:
# Fit the model
import time 
print("Sampling prior...")
start = time.time()
mmm.sample_prior(500)
print(f"Prior sampling took {time.time() - start:.1f} seconds")

start = time.time()
mmm.sample_posterior(n_chains=4, n_adapt=500, n_burnin=500, n_keep=1000)
print(f"Posterior sampling took {time.time() - start:.1f} seconds")

print("\n✓ Model fitting complete!")

In [ ]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()

In [ ]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

In [ ]:
filename = 'meridian_summary_market_clean_data_control_none.html'
outputpath = 'output/'
mmm_summarizer = summarizer.Summarizer(mmm)
mmm_summarizer.output_model_results_summary(filename=filename, filepath=outputpath)


In [ ]:
print("Media,   Prob ROAS >= 0")
list(zip(media_to_channel.values(), 
         (mmm.inference_data.posterior.roi_m >=1).mean(
             dim=('chain', 'draw')).values))

In [ ]:
print("Channel -- Posterior -- Prior")
prior_v_posterior = pd.DataFrame(list(zip(media_to_channel.values(),
                                          (mmm.inference_data.posterior.roi_m).mean(dim=('chain', 'draw')).values,
                                          mmm.inference_data.prior.roi_m.mean(dim=('chain','draw')).values)),
                                          columns = ['media', 'posterior_mean', 'prior_mean'])
print(prior_v_posterior)